# 第11回　教師なし学習
***
> **前提**: 第6回までの教師あり学習に続き，ラベルなしデータの分析を学びます。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. k-means クラスタリング
2. クラスタ数の評価
3. PCA による次元削減
4. 2次元可視化

---

## この回で学ぶこと

### 教師あり学習 vs 教師なし学習

これまでの学習（第1〜10回）は，すべて「正解ラベル」がある**教師あり学習**だった。

```
教師あり学習: データ + 正解ラベル → モデルが「入力→出力」の関係を学習
教師なし学習: データのみ           → データ内の隠れた構造・パターンを発見
```

現実には「正解ラベルがない」データの方が圧倒的に多い。センサーデータ，SNS投稿，画像，遺伝子発現データなど，ラベル付けコストが高いデータには教師なし学習が不可欠だ。

### k-means クラスタリング

k-means の仕組みを直感的に理解しよう：

1. ランダムにk個の「重心（centroid）」を配置
2. 各データ点を最も近い重心に割り当て（クラスタを形成）
3. 各クラスタの重心を再計算
4. 変化がなくなるまで2〜3を繰り返す

**重要な特性**：
- 初期値依存性：`random_state` を固定しないと毎回結果が異なる
- k は事前に決める必要がある（最大の欠点）
- 球形クラスタ向け：細長い，三日月形などのクラスタは苦手
- スケールに敏感：`StandardScaler` で正規化が必須

### シルエットスコアとは

シルエットスコアは「各データ点が自分のクラスタに適切に属しているか」を -1〜1 の値で評価する指標だ：
- **値が1に近い**: そのデータ点は自分のクラスタの中心に近く，他のクラスタから遠い → 良い分類
- **値が0に近い**: クラスタの境界付近にある
- **値が負**: 本来は別のクラスタに属すべきかもしれない

### 主成分分析（PCA）とは

高次元データ（例：30変数）を低次元（2次元）に圧縮する手法だ。単純に変数を削除するのではなく，**データの分散が最大になる方向（主成分）を見つけ**，その方向に投影する。

活用場面：
- **可視化**: 30次元のデータを2次元にして散布図で確認
- **前処理**: 次元を減らして計算コストを削減，ノイズ除去
- **多重共線性の解決**: 相関する変数をまとめる

> **卒業研究での活用例**: テキストデータの単語ベクトル（数千次元）を PCA で2〜3次元に圧縮し，似た意味の単語をクラスタリングする研究はよく見られる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris, load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


## 問題1　k-means クラスタリング　【説明】
***

### Iris データセットについて

Iris データセットは機械学習の入門で最もよく使われるデータセットだ：
- 150件のアヤメの花のデータ
- 4特徴量：がく片の長さ/幅，花びらの長さ/幅（単位：cm）
- 3クラス：Setosa / Versicolor / Virginica

今回は「正解ラベル（品種）を使わずに」，特徴量だけから3つのグループを自動発見できるかを試す。発見したクラスタが実際の品種と一致すれば，この手法の有効性が確認できる。

### なぜ標準化が必要か

Iris の4特徴量はどれもcm単位で似たスケールだが，k-means はユークリッド距離を使うため，スケールが大きい変数の影響が過大になる。**標準化は k-means の前処理として必須**だと覚えておこう。

### クラスタラベルの注意点

k-means のラベルは 0, 1, 2 が割り当てられるが，この番号と品種の名前は対応していない。例えば「クラスタ0 = Setosa」とは限らず，実行するたびに対応が変わりうる（番号に意味はない）。

### 課題

下のコードセルは、Iris を標準化して `KMeans` でクラスタリングし、シルエットスコアを出すところまで **完成済み**です。今回はコードを書くのではなく、**1行ずつ「何をしているか」を読み解く**のが目的です。

各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しないこと）。書き終えたらセルを実行し、エラーなくシルエットスコアが出ることを確認してください。

説明を書くときは、次の問いを意識してください：

- なぜ k-means の前に `StandardScaler` で標準化するのか？（k-means が使う「距離」と関係づけて）
- `fit_predict` は「学習」と「割り当て」のどちらも行うが、具体的に何を返しているのか？
- `silhouette_score` の値が高い／低いと、クラスタリングについて何が言えるのか？

> **設計判断（一言で）**: `KMeans(n_clusters=3, ...)` の `3` はどこから来た数字でしょうか。Iris の正解ラベルを「使わずに」この `3` を決めるにはどうすればよいか、解答用コードセルに1〜2文で書いてください（次の問題2の伏線です）。

In [ ]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# 説明は AI に書かせず、自分で書くこと

iris = load_iris()                                        # 説明:
X_iris = iris.data                                        # 説明:（形状は (150, 4)）

scaler = StandardScaler()                                 # 説明:（なぜ標準化が必要？）
X_scaled = scaler.fit_transform(X_iris)                   # 説明:（fit_transform は何をしている？）

kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)  # 説明:（n_clusters=3 と random_state の意味）
labels = kmeans.fit_predict(X_scaled)                     # 説明:（fit_predict は何を返す？）

print("先頭10件のクラスタラベル:", labels[:10])           # 説明:
score = silhouette_score(X_scaled, labels)                # 説明:（引数の順番と返り値の意味）
print(f"シルエットスコア: {score:.4f}")


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) なぜ k-means の前に標準化するのか（k-means が使う「距離」と関連づけて）
answer_1_a = """
"""

# (1-b) `fit_predict` は具体的に何を返しているか
answer_1_b = """
"""

# (1-c) `silhouette_score` が高い／低いとクラスタリングについて何が言えるか
answer_1_c = """
"""

# (1-d) 設計判断：`n_clusters=3` の `3` はどこから来たか／正解ラベルを使わずに決めるには（1〜2文）
answer_1_d = """
"""



## 問題2　最適なクラスタ数の探索　【骨格+選択+実験】
***

### k をどう決めるか

k-means の最大の課題は「k（クラスタ数）を事前に決めなければならない」点だ。データを見ても適切な k が分からない場合，以下の手法で探索する：

**① エルボー法（Elbow Method）**
- 各 k の「クラスタ内誤差の合計（inertia）」をプロット
- グラフが「肘（elbow）」のように急に折れ曲がる点が最適 k の目安
- `kmeans.inertia_` で取得できる

**② シルエットスコア法**
- 各 k のシルエットスコアをプロット
- **スコアが最も高い k** が最適（今回使用）

> **どちらを使うべき?** 2つの方法が一致する k があれば信頼性が高い。一致しない場合はドメイン知識（そのデータについての専門知識）も参考にする。

### 課題

下のコードセルは、k を変えながら **inertia（エルボー法）** と **シルエットスコア** を計算し、2つの図を描くところまで **完成済み**です。教師なし学習には「正解ラベル」がないので、**この2つの図を根拠に自分で k を決める**のがこの問題の核心です。

**(1) 選択**：図を見て、クラスタ数 k を1つ選んでください。
- エルボー法では「折れ線が急に折れ曲がる（肘の）点」が目安
- シルエットスコアでは「値が最大になる k」が目安
- 2つが一致すれば信頼性が高い。一致しないときは**どちらをなぜ優先したか**を解答用コードセルに書く

**(2) 実験**：`k_range`（★印の行）を変えて **最低5通り**の k を試し、各 k の `inertia` と `silhouette` を **✍️ 解答用コードセルの実験ログ**に記録してください。なお、ループ内の **クラスタリングの核心1行はあなたが書きます**（`# ★あなたが書く★`）。

> **考察**: 教師あり学習と違い、教師なし学習には accuracy のような「正解との一致率」がありません。それでも k の良し悪しを判断できるのはなぜですか？ `inertia`（小さいほど良いが k を増やせば必ず下がる）と `silhouette`（クラスタの分離の良さ）の**意味の違い**に触れて、解答用コードセルに書いてください。


In [ ]:
# === 完成済みコード：実行して2つの図（エルボー法／シルエット）を観察してください ===
# 問題1 で作った X_scaled（標準化済み Iris）を再利用します

# === ★ここを変えて実験する★：試すクラスタ数の範囲 ===
k_range = range(2, 7)   # 2,3,4,5,6 を試す（範囲を変えて実験してよい）

inertias = []
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    # ★あなたが書く★：km で X_scaled をクラスタリングし、各点のラベルを lab に求める（1行）
    #   ヒント: fit_predict(データ) が「学習＋割り当て」をまとめて行いラベル配列を返す
    lab = ___
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lab))
    print(f"k={k}: inertia={km.inertia_:8.2f}  silhouette={silhouettes[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), inertias, "o-")
axes[0].set_title("エルボー法（inertia）")
axes[0].set_xlabel("クラスタ数 k")
axes[0].set_ylabel("inertia（クラスタ内誤差の合計）")
axes[1].plot(list(k_range), silhouettes, "o-", color="green")
axes[1].set_title("シルエットスコア")
axes[1].set_xlabel("クラスタ数 k")
axes[1].set_ylabel("silhouette_score")
plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(silhouettes))]
print("シルエットスコアが最大の k =", best_k)


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 実験ログ（5通り以上の k を記入）
experiment_log = pd.DataFrame([
    {'row': 1, 'k': 2, 'inertia': None, 'silhouette': None},
    {'row': 2, 'k': 3, 'inertia': None, 'silhouette': None},
    {'row': 3, 'k': 4, 'inertia': None, 'silhouette': None},
    {'row': 4, 'k': 5, 'inertia': None, 'silhouette': None},
    {'row': 5, 'k': 6, 'inertia': None, 'silhouette': None},
    {'row': 6, 'k': None, 'inertia': None, 'silhouette': None},
])

# (2-b) 選択：図を見て決めたクラスタ数 k：(　)
# 例: 3  （シルエットが最大だった k の数値）
answer_2_b = ""

# (2-c) その根拠（エルボー法／シルエットのどちらをどう読んだか。食い違った場合はどちらをなぜ優先したか）
answer_2_c = """
"""

# (2-d) 考察：正解ラベルが無いのに k の良し悪しを判断できる理由／inertia と silhouette の意味の違い
answer_2_d = """
"""



## 問題3　PCA による次元削減　【骨格+実験】
***

### Wine データセットについて

- 3種類のワインの化学分析データ（178件）
- 13特徴量：アルコール度数，リンゴ酸，灰分，マグネシウム，フェノール類，フラボノイドなど
- 多くの特徴量が互いに相関している（例：フェノール類とフラボノイド）

PCA は相関する変数をうまく圧縮するのが得意だ。13次元の情報をどれだけ2次元で保持できるか確認しよう。

### 寄与率（説明分散比）の意味

`pca.explained_variance_ratio_` は各主成分が「元データの全分散のうち何%を説明するか」を示す：
- 第1主成分：データの変動が最も大きい方向
- 第2主成分：第1主成分と直交する方向で，次に変動が大きい方向

```
例: explained_variance_ratio_ = [0.36, 0.19]
→ 第1主成分で 36%, 第2主成分で 19% の情報を保持
→ 合計 55% の情報で 13 次元 → 2 次元に圧縮
```

**累積寄与率**（合計）が70〜80%以上あれば，2次元での可視化に十分な情報が保持されている。

### PCA の前に標準化が必須な理由

PCA は「分散が大きい方向」を主成分として選ぶ。スケールの大きい特徴量（例：マグネシウム：70〜162mg）は自動的に「重要な方向」として選ばれてしまう。標準化することで，すべての特徴量を公平に扱える。

### 課題

下のコードセルは、Wine データ（13次元）を標準化し、`n_components` を変えながら **累積寄与率（`explained_variance_ratio_` の合計）** を表示するところまで **完成済み**です。「13次元を何次元まで圧縮すると、元の情報を何%保てるか」を**実験で確かめる**のが目的です。

`n_components_list`（★印の行）を変えて **最低5通り**試し、各 `n_components` の**累積寄与率**を **✍️ 解答用コードセルの実験ログ**に記録してください。なお、ループ内の **PCA 学習の核心1行はあなたが書きます**（`# ★あなたが書く★`）。

そのうえで答えてください：

> **設計判断**: 「可視化のために2次元に落とす」のと「情報を80%以上保ったまま次元を減らす」のとでは、選ぶべき `n_components` が変わります。**この Wine データで累積寄与率80%を超えるのは何次元からか**を実験ログから読み取り、解答用コードセルに書いてください。
>
> **考察**: 単純に「変数を13個から2個に削る（列を捨てる）」のと、PCA で「2次元に圧縮する」のは何が違いますか？ PCA が**相関した変数をまとめる**点に触れて説明してください。

In [ ]:
# === 完成済みコード：n_components を変えて累積寄与率を観察してください ===
wine = load_wine()
X_wine_scaled = StandardScaler().fit_transform(wine.data)
print("元データの次元数:", X_wine_scaled.shape[1])  # 13

# === ★ここを変えて実験する★：試す主成分の数（5通り以上） ===
n_components_list = [1, 2, 3, 5, 10, 13]

for n in n_components_list:
    pca = PCA(n_components=n)
    # ★あなたが書く★：pca を標準化済みデータ X_wine_scaled で学習する（1行）
    #   ヒント: pca.fit(データ)
    ___
    cum = pca.explained_variance_ratio_.sum()
    print(f"n_components={n:2d} -> 累積寄与率={cum:.4f}（{cum * 100:.1f}%）")

# 参考：全成分の累積寄与率カーブ（何次元で何%か一目で分かる）
pca_full = PCA().fit(X_wine_scaled)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, len(cumsum) + 1), cumsum, "o-")
plt.axhline(0.8, color="red", linestyle="--", label="80% ライン")
plt.xlabel("主成分の数")
plt.ylabel("累積寄与率")
plt.title("累積寄与率カーブ（Wine）")
plt.legend()
plt.show()


In [ ]:
# === ✍️ 問題3 解答（採点対象）===
import pandas as pd


# (3-a) 実験ログ（`n_components` と累積寄与率。5通り以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'n_components': 1, 'cumulative_variance_pct': None},
    {'row': 2, 'n_components': 2, 'cumulative_variance_pct': None},
    {'row': 3, 'n_components': 3, 'cumulative_variance_pct': None},
    {'row': 4, 'n_components': 5, 'cumulative_variance_pct': None},
    {'row': 5, 'n_components': 10, 'cumulative_variance_pct': None},
    {'row': 6, 'n_components': 13, 'cumulative_variance_pct': None},
])

# (3-b) 設計判断：累積寄与率80%を超えるのは何次元からか
answer_3_b = """
"""

# (3-c) 考察：「列を捨てる」のと PCA で「2次元に圧縮する」の違い（相関した変数をまとめる点）
answer_3_c = """
"""



## 問題4　PCA 結果の可視化と解釈　【説明+選択】
***

### 可視化で何を確認するか

PCA 後の2次元散布図を色分けして描くことで：
1. **クラスが分離できているか**：色が綺麗に分かれていれば，PCA が識別に有効な方向を捉えている
2. **外れ値の存在**：散布図の端に孤立した点がないか
3. **クラス間の重なり**：重なりが多い場合，このデータは2次元では分離が難しい

### この可視化の意義

もし3クラスが2次元平面上で綺麗に分離できていれば，わずか2つの主成分（元の13特徴量の線形結合）が品種分類に十分な情報を持っていることを意味する。これは：
- 教師なし学習（今回）：どのクラスタに属するかを探索的に確認
- 次元削減の前処理として：k-means クラスタリングの前に PCA を適用することもある

### 課題

下のコードセルは、Wine を `PCA(n_components=2)` で2次元に圧縮し、正解ラベルで色分けした散布図を描くところまで **完成済み**です。

**(1) 説明**：各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しない）。特に `fit_transform` が「13次元 → 2次元」に何をしているのか、`explained_variance_ratio_` が何を表すのかを意識してください。

**(2) 選択**：教師なし学習には「正解ラベル」がないのが普通です。今回は確認のため `target` で色分けしますが、**ラベルが無いとき**にクラスタリングや次元削減の「良し悪し」をどう判断すべきか、次から**最も主要な根拠を1つ選び、理由**を解答用コードセルに書いてください。

> - **(A) accuracy（正解率）で測る**
> - **(B) シルエットスコアで「クラスタの分離の良さ」を測る**
> - **(C) PCA の累積寄与率で「保持できた情報量」を測る**
> - **(D) 散布図を人が見て、まとまり・重なり・外れ値を解釈する**
> - **(E) Davies-Bouldin 指数など別のクラスタ評価指標で測る**
> - **(F) inertia（クラスタ内誤差）だけを見て小さいほど良いと判断する**
>
> ヒント：「正解ラベルが無い」前提では使えない指標が混ざっています（例えば (A) は使えません）。(F) のように「小さいほど良いが k を増やせば必ず下がる」指標を単独で使う落とし穴にも注意。今回の目的（クラスタの良さを見るのか、次元削減で情報がどれだけ残ったかを見るのか）によっても答えは変わります。**なぜそれを主要な根拠にするのか**を書いてください。

> **考察**: もし3クラスが2次元平面で綺麗に分かれていたら、それは元の13特徴量について何を意味しますか？（「2つの主成分＝13特徴量の線形結合」という観点で）


In [ ]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# wine / X_wine_scaled は問題3で用意済み。ここでは2次元に圧縮して可視化します。

pca2 = PCA(n_components=2)                            # 説明:（n_components=2 にする目的は？）
X_pca = pca2.fit_transform(X_wine_scaled)            # 説明:（13次元 → 2次元 に何をしている？）
y = wine.target                                      # 説明:（色分けに使う「正解ラベル」）

ev = pca2.explained_variance_ratio_                  # 説明:（各主成分の寄与率）
print("第1/第2主成分の寄与率:", ev.round(3), " 累積:", round(ev.sum(), 3))

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette="Set1")  # 説明:（hue=y の役割）
plt.xlabel(f"第1主成分（寄与率 {ev[0] * 100:.1f}%）")
plt.ylabel(f"第2主成分（寄与率 {ev[1] * 100:.1f}%）")
plt.title("Wine データの PCA 2次元可視化")
plt.legend(title="品種")
plt.show()


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (4-a) 選択：ラベルが無いときに良し悪しを判断する主要な根拠（A〜F）：(　)
# (A) accuracy（正解率）で測る
# (B) シルエットスコアでクラスタの分離の良さを測る
# (C) PCA の累積寄与率で保持できた情報量を測る
# (D) 散布図を人が見て、まとまり・重なり・外れ値を解釈する
# (E) Davies-Bouldin 指数など別のクラスタ評価指標で測る
# (F) inertia だけを見て小さいほど良いと判断する
answer_4_a = ""

# (4-b) その理由（なぜそれを主要な根拠にするか／今回の目的と結びつけて）
answer_4_b = """
"""

# (4-c) 考察：3クラスが2次元で綺麗に分かれていたら、元の13特徴量について何が言えるか
answer_4_c = """
"""

